# 4.0 — End-to-End Model Comparison
# Tests all three training stages on the 2 questions used across the pipeline:
#   Q1: "What is BPMN and what is its primary goal?"
#   Q2: "Describe all five Gateway types in BPMN 2.0 and when to use each."


In [1]:
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# The 2 questions used in instruction fine-tuning (indices 0 and 16)
questions = [
    "What is BPMN and what is its primary goal?",
    "Describe all five Gateway types in BPMN 2.0 and when to use each.",
]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def get_latest_checkpoint(ckpt_dir):
    ckpts = sorted(
        [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[1])
    )
    return os.path.join(ckpt_dir, ckpts[-1])

def generate(model, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.2,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Setup complete. Device:", device)


c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete. Device: cpu


In [2]:
# --- Stage 1: Non-Instruction Model (1.0) ---
# Trained on raw BPMN text, no instruction format. Prompt = plain question.
ckpt_1 = "./tinyllama-lora/checkpoint-370"
base_1 = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float32, low_cpu_mem_usage=True)
non_instruction_model = PeftModel.from_pretrained(base_1, ckpt_1).merge_and_unload().to(device)
non_instruction_model.eval()
print("Non-instruction model loaded from:", ckpt_1)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15729.81it/s]


Non-instruction model loaded from: ./tinyllama-lora/checkpoint-370


In [3]:
print("=" * 70)
print("STAGE 1 — Non-Instruction Model")
print("=" * 70)
for q in questions:
    print(f"\nQuestion: {q}")
    print("Output:", generate(non_instruction_model, q))
    print("-" * 70)


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAGE 1 — Non-Instruction Model

Question: What is BPMN and what is its primary goal?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output: 207 Business Process Model and Notation, v2.0 The purpose of the BPMN model is to provide a standardized way for describing business processes that can be used by process engineers in order to develop and maintain them. This standardization allows companies to reuse existing processes without having to re-engineer them or change their underlying technology. In addition, it provides an open platform for developing new processes based on the same principles as those already defined. It also enables organizations to share these processes with other organizations through the use of standards such as XML Schemas (e.g., XSD) and WS-BPEL .
----------------------------------------------------------------------

Question: Describe all five Gateway types in BPMN 2.0 and when to use each.
Output: 
----------------------------------------------------------------------


In [4]:
# --- Stage 2: Instruction Fine-Tuned Model (2.0) ---
# Fine-tuned on instruction format. Prompt must use ### Instruction / ### Response template.
ckpt_2 = get_latest_checkpoint("./tinyllama-instruction")
base_2 = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float32, low_cpu_mem_usage=True)
instruction_model = PeftModel.from_pretrained(base_2, ckpt_2).to(device)
instruction_model.eval()
print("Instruction model loaded from:", ckpt_2)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3321.25it/s]


Instruction model loaded from: ./tinyllama-instruction\checkpoint-100


In [5]:
print("=" * 70)
print("STAGE 2 — Instruction Fine-Tuned Model")
print("=" * 70)
for q in questions:
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    print(f"\nQuestion: {q}")
    print("Output:", generate(instruction_model, prompt))
    print("-" * 70)


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAGE 2 — Instruction Fine-Tuned Model

Question: What is BPMN and what is its primary goal?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output: BPMN stands for Business Process Modeling Notation. It is a OMG standard (version 2.0) for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. A secondary but equally important goal is to ensure that XML-based execution languages, such as WS-BPEL (Web Services Business Process Execution Language), can be visualized in a user-friendly notation. BPMN creates a standard bridge between business process implementation technologies. A third but related goal is to facilitate the auditing of business processes. Finally, BPMN provides a standardized mechanism for linking business processes with business intelligence tools, so that business users can monitor their processes more effectively.
A primary goal is to provide a notation that is readily under

#Testing with Instruction-Fine-Tuned Model

In [9]:
# --- Stage 3: DPO Preference-Aligned Model (3.0) ---
# Built on top of 2.0. Load: base → merge instruction LoRA → apply DPO LoRA.
ckpt_2_path = get_latest_checkpoint("./tinyllama-instruction")
ckpt_3_path = "./tinyllama-dpo-step57"  # adapter files are at the root, no checkpoint-* subdirs

base_3 = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float32, low_cpu_mem_usage=True)
merged_3 = PeftModel.from_pretrained(base_3, ckpt_2_path).merge_and_unload()

# Save + reload to clear PEFT internals before applying DPO adapter
TEMP_PATH = "./tinyllama-merged-for-test"
merged_3.save_pretrained(TEMP_PATH)
clean_3 = AutoModelForCausalLM.from_pretrained(TEMP_PATH, torch_dtype=torch.float32, low_cpu_mem_usage=True)
dpo_model = PeftModel.from_pretrained(clean_3, ckpt_3_path).to(device)
dpo_model.eval()
print("DPO model loaded from:", ckpt_3_path)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2636.01it/s]


DPO model loaded from: ./tinyllama-dpo-step57


In [10]:
print("=" * 70)
print("STAGE 3 — DPO Preference-Aligned Model")
print("=" * 70)
for q in questions:
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    print(f"\nQuestion: {q}")
    print("Output:", generate(dpo_model, prompt))
    print("-" * 70)


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAGE 3 — DPO Preference-Aligned Model

Question: What is BPMN and what is its primary goal?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output: BPMN stands for Business Process Modeling Notation. It is a OMG standard (version 2.0) for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. A secondary but equally important goal is to ensure that XML-based execution languages, such as WS-BPEL (Web Services Business Process Execution Language), can be visualized in a user-friendly notation. BPMN creates a standardized bridge between business process design and process implementation. A third but related goal is to facilitate the interoperability of business processing systems. Finally, BPMN provides a standardized mechanism for monitoring business processes. An example of this last use case is found in the Health Insurance Portability and Accountability Act (HIPAA), which
-----------------

##Testing with DPO (Preference-Aligned) Model